# 🚀 第 46 课 · Attention 性能

> 算术强度与 roofline;naive vs 分块 vs torch SDPA 的 GPU 实测对比;看懂 memory-bound 的含义

**章节**: 第 7 章 · Attention Kernel 实战

---

> **📚 本笔记本属于《minivllm: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 定义算术强度(arithmetic intensity = FLOPs/字节)并理解 roofline 模型与 ridge 点
- 在 GPU 上实测 naive / 分块 / torch SDPA 三种 attention 的耗时,控制变量
- 用 pyecharts 柱状图 + plotly 折线图对比不同序列长度的性能
- 理解 memory-bound(访存受限)vs compute-bound(算力受限)的判断方法
- 代入 LLaMA/Qwen 真实规模算 attention 的算术强度,判断它为何 memory-bound
- 理解减少访存是 attention 优化的关键,以及它对 vLLM 后端的启示

## 📑 本课目录

1. **直觉:邮局分拣的两种瓶颈** — 带宽 vs 算力,谁先堵
2. **算术强度与 roofline 模型** — FLOPs/字节 与 ridge 点,判断 bound 的公式
3. **三种实现的耗时对比** — naive / 分块 / torch SDPA,同条件 GPU 实测
4. **pyecharts 柱状图 + plotly 曲线** — 一张图看懂三者的差距与斜率
5. **真实规模:attention 为何 memory-bound** — 代入 LLaMA-7B 算算术强度
6. **与 vLLM 的关系** — 后端选择关注 memory-bound;FP8 KV cache 的启示
7. **小结 + 练习 + 延伸阅读** — 要点、动手题、论文链接

## 🔗 参考资料

- [Roofline: An Insightful Visual Performance Model (CACM 2009)](https://people.eecs.berkeley.edu/~pattrsn/mytalks/Roofline.pdf)
- [FlashAttention (IO-aware)](https://arxiv.org/abs/2205.14135)
- [PyTorch scaled_dot_product_attention](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

## 1. 直觉:邮局分拣的两种瓶颈 🏤

邮局分拣信件,可能遇到两种瓶颈:

- **人不够**(算力不够):每人每秒能处理的信件数有限,排队等处理 → **compute-bound(算力受限)**;
- **传送带太窄**(带宽不够):人手绰绰有余,但信件搬不上传送带,机器大部分时间在**等信件** → **memory-bound(访存受限)**。

现代 GPU 的算力(每秒几十万亿次浮点运算)通常远大于访存带宽,所以很多算子其实是**带宽不够**,不是算力不够。attention 尤其如此——它的问题从来不是「算得慢」,而是「**搬得慢**」。

✅ 每课第一段代码:设置 KMP 保护、固定 seed。本课是**核心真机对比课**——naive/分块/SDPA 全部在 GPU 上同步计时,torch.cuda.synchronize 保证测的是真实耗时。

In [ ]:
# -*- coding: utf-8 -*-
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows OMP 库冲突防护
import math, time, torch, numpy as np

dev = "cuda" if torch.cuda.is_available() else "cpu"     # 有 CUDA 就跑真实 GPU（RTX 5060）
torch.manual_seed(0)
print("torch  :", torch.__version__)
print("device :", dev, "| GPU:", torch.cuda.get_device_name(0) if dev == "cuda" else "无(CPU)")
print("num_threads :", torch.get_num_threads())
print("SDPA flash 可用         :", torch.backends.cuda.flash_sdp_enabled())
print("SDPA mem_efficient 可用 :", torch.backends.cuda.mem_efficient_sdp_enabled())

## 2. 算术强度与 roofline 模型 📐

一个算子是 compute-bound 还是 memory-bound,取决于**每字节数据要配多少浮点运算**,即**算术强度**:

$$ \text{AI} = \frac{\text{FLOPs(计算量)}}{\text{Bytes(访存量)}} \quad [\text{FLOPs/字节}] $$

硬件的「临界强度(ridge point)」是算力与带宽之比([Williams et al., CACM'09](https://people.eecs.berkeley.edu/~pattrsn/mytalks/Roofline.pdf)):

$$ \text{AI}^\star = \frac{C_{\text{peak}}(\text{算力 TFLOPs/s})}{BW_{\text{HBM}}(\text{带宽 TB/s})} $$

判断规则:

| 条件 | 类型 | 性能上界 |
|---|---|---|
| $\text{AI} < \text{AI}^\star$ | **memory-bound** | 带宽屋顶 $BW \cdot \text{AI}$ |
| $\text{AI} > \text{AI}^\star$ | compute-bound | 算力屋顶 $C_{\text{peak}}$ |

典型数值:A100 ridge ≈ 156–200 FLOPs/字节,H100 ≈ 295 FLOPs/字节。而 naive attention 的算术强度只有约 **4** FLOPs/字节(见第 5 节推导),深陷 memory-bound 区。

> 📄 参考口径:attention 里一个 $N\times N$ 元素要被多次读写,而真正的计算量相对有限,导致算术强度低,瓶颈落在访存上。这也是 FlashAttention 标题里「IO-aware」的由来。

## 3. 三种实现的耗时对比 ⏱️

先写出 naive 与分块两版(与第 44 课一致),再加 torch 内置的 `scaled_dot_product_attention`(SDPA)。在**相同输入**下用 `torch.cuda.synchronize()` 精确计时——GPU 是异步的,不同步就会只量到「提交」的时间。

**naive_attention**:三步把大张量写进内存,O(N²) 访存。

In [2]:
def naive_attention(Q, K, V):
    """朴素注意力:把 QK^T / softmax / 加权和 三个大张量全部写进 HBM。
    Q, K, V: (B, H, N, d)。返回与输入同形状的输出 O。
    """
    scale = 1.0 / math.sqrt(Q.shape[-1])          # 1/√d 缩放因子
    S = torch.einsum("bhnd,bhmd->bhnm", Q, K) * scale   # ① QK^T: (B,H,N,N)
    P = torch.softmax(S, dim=-1)                  # ② softmax: (B,H,N,N)
    O = torch.einsum("bhnm,bhmd->bhnd", P, V)     # ③ 加权和: (B,H,N,d)
    return O

**flash_attention_chunked**:分块 + online softmax,减少中间张量落盘。

In [3]:
def flash_attention_chunked(Q, K, V, block_M=64):
    """分块 FlashAttention(torch 手写版):online softmax + K/V 分块扫描。
    只在寄存器/片上保存 running max(m)、running sum(l) 与累加器 O,
    大张量 S/P 从不整体落进 HBM——这就是 FlashAttention 的核心思想。
    Q, K, V: (B, H, N, d)。block_M 控制 K/V 方向上每次读入的块宽。
    """
    B, H, N, d = Q.shape
    O = torch.zeros_like(Q)                                   # 输出累加器
    m = torch.full((B, H, N, 1), float("-inf"), device=Q.device)  # 逐行 running max
    l = torch.zeros((B, H, N, 1), device=Q.device)                # 逐行 running sum
    scale = 1.0 / math.sqrt(d)
    for j in range(0, N, block_M):                            # 沿 K/V 分块扫描
        Kj = K[:, :, j:j + block_M, :]                        # 读一块 K (B,H,Bm,d)
        Vj = V[:, :, j:j + block_M, :]                        # 读一块 V (B,H,Bm,d)
        S = torch.einsum("bhnd,bhmd->bhnm", Q, Kj) * scale    # 块内打分 (B,H,N,Bm)
        m_new = torch.maximum(m, S.max(dim=-1, keepdim=True).values)   # 更新 running max
        P = torch.exp(S - m_new)                              # 以新 max 归一(防溢出)
        l_new = l * torch.exp(m - m_new) + P.sum(dim=-1, keepdim=True)  # 更新 running sum
        O = O * torch.exp(m - m_new) + torch.einsum("bhnm,bhmd->bhnd", P, Vj)  # 校正旧累加
        m = m_new                                             # 滚到下一块
        l = l_new
    return O / l                                              # 最终除以分母

🎯 同条件、同 GPU、同步计时,结果最诚实:小 N 时大家都被 kernel 启动开销吃住;N 拉长后 SDPA 明显更快;torch 手写分块因每块都要 Python 循环而反被拖慢。三者数值几乎一致(误差 ~1e-6)。

In [ ]:
# 同条件、同步计时,对比三种实现的真实耗时
import torch.nn.functional as F

def bench(fn, *a, iters=5):
    for _ in range(2):            # warmup:触发编译/缓存
        fn(*a)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(*a)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000   # 毫秒

torch.manual_seed(0)
B, H, N, d = 1, 4, 1024, 64
Q = torch.randn(B, H, N, d, device=dev)
K = torch.randn(B, H, N, d, device=dev)
V = torch.randn(B, H, N, d, device=dev)
print(f"输入: Q/K/V shape = {tuple(Q.shape)} @ {dev}")

t_naive = bench(naive_attention, Q, K, V)
t_flash = bench(flash_attention_chunked, Q, K, V, 64)
t_sdpa  = bench(lambda q, k, v: F.scaled_dot_product_attention(q, k, v), Q, K, V)
print(f"  naive : {t_naive:7.3f} ms")
print(f"  分块  : {t_flash:7.3f} ms")
print(f"  SDPA  : {t_sdpa:7.3f} ms")
print(f"  SDPA vs naive 加速 : {t_naive / max(t_sdpa, 1e-9):6.2f} x")

o1 = naive_attention(Q, K, V); o2 = F.scaled_dot_product_attention(Q, K, V)
print(f"  naive vs SDPA 最大误差 = {(o1 - o2).abs().max().item():.2e}")
torch.cuda.empty_cache()

## 4. 真机扫描:随 N 的耗时曲线 📊

在 GPU 上扫多个序列长度 $N$,逐个量出三种实现的真实耗时。naive 受 O(N²) 访存支配会**快速向上翘**;分块与 SDPA 把它们留在片上,增长平缓得多。先跑出数字、打印成表,再画图。

🎯 栏目:naive / 分块 / SDPA。红色 naive 随 N 几乎按 O(N²) 蹿升,内核化的 SDPA 涨得最慢——真正意义上的「快」来自融合内核(SDPA),不是 torch 的 Python 循环。

In [ ]:
# GPU 上扫 N,打印三种实现的耗时表
from pyecharts.charts import Bar
from pyecharts import options as opts

Ns = [256, 512, 1024, 2048, 4096]
res = []
print(f"{'N':>6} {'naive':>10} {'分块':>10} {'SDPA':>10}   (ms)")
for n in Ns:
    q = torch.randn(1, H, n, d, device=dev)      # (B,H,N,d)
    k = torch.randn(1, H, n, d, device=dev)
    v = torch.randn(1, H, n, d, device=dev)
    tn = bench(naive_attention, q, k, v)
    tf = bench(flash_attention_chunked, q, k, v, 64)
    ts = bench(lambda a, b, c: F.scaled_dot_product_attention(a, b, c), q, k, v)
    res.append([round(tn, 3), round(tf, 3), round(ts, 3)])
    print(f"{n:>6} {tn:>10.3f} {tf:>10.3f} {ts:>10.3f}")
    torch.cuda.empty_cache()
print("\n→ naive 把 S/P 整体写进 HBM(N² 访存)最贵;内核化的 SDPA 最省;torch 分块虽省显存但 Python 循环拖慢。")

🎨 红色线(naive)向上翘(受 O(N²) 访存支配),绿色 SDPA 最平、胜在融合内核;蓝色 torch 分块介于中间。快慢与数值(逐点 <1e-6)无关,差别全在访存与内核融合。

In [ ]:
# 用 pyecharts 柱状图 + plotly 折线图可视化
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.graph_objects as go

# 柱状图:同 N 下三种实现并排
bar = (Bar()
       .add_xaxis([f"N={n}" for n in Ns])
       .add_yaxis("naive(ms)", [r[0] for r in res], color="#E45756")
       .add_yaxis("分块(ms)", [r[1] for r in res], color="#4C78A8")
       .add_yaxis("SDPA(ms)", [r[2] for r in res], color="#72B7B2")
       .set_global_opts(title_opts=opts.TitleOpts(title=f"三种 attention 耗时对比(GPU, H={H}, d={d})"),
                        xaxis_opts=opts.AxisOpts(name="序列长度"),
                        yaxis_opts=opts.AxisOpts(name="耗时(ms)")))
bar.render_notebook()

# 折线图:看斜率差异
tn = [r[0] for r in res]; tf = [r[1] for r in res]; ts = [r[2] for r in res]
fig = go.Figure()
for name, arr, col in [("naive", tn, "#E45756"), ("分块", tf, "#4C78A8"), ("SDPA", ts, "#72B7B2")]:
    fig.add_trace(go.Scatter(x=Ns, y=arr, name=name, mode="lines+markers",
                             line=dict(color=col, width=2.5)))
fig.update_layout(title=f"三种实现耗时随序列长度变化(GPU, H={H}, d={d})",
                  xaxis_title="序列长度 N", yaxis_title="耗时(ms)", height=440,
                  legend=dict(orientation="h", y=1.12), margin=dict(l=10, r=10, t=60, b=10))
fig

## 5. 真实规模:attention 为何 memory-bound ⚖️

用 LLaMA-7B 的真实配置算一遍 attention 的算术强度。设 $H=32$ 头、$d=128$、序列 $N=4096$:

- **计算量**:$QK^\top$ 与 $PV$ 各约 $2N^2 d$ → $\text{FLOPs} \approx 4N^2 d = 4\cdot 4096^2 \cdot 128 \approx 8.6\,\text{G}$;
- **访存量(bf16)**:$Q,K,V,O$ 各 $N\cdot d$ 个元素($\times 2$ 字节),加上 $S,P$ 各 $N^2$ 个元素($\times 2$)读写
  → $\text{bytes} \approx 8Nd + 4N^2$(约) $\approx 2\,\text{GB}$;
- **算术强度**:$\text{AI} \approx \frac{4N^2d}{4N^2 + 8Nd} \approx \frac{4\cdot 128 \cdot N}{4N + 8\cdot 128} \approx 120$(大 N 时趋近 $d$ 量级)。

把上面推导写成代码算精确值:

🎯 关键结论:naive attention 的 AI(~120)远低于 H100 的 ridge(295),所以是 memory-bound——增加算力没用,减少访存才是关键。FlashAttention 把 AI 提到几百,才逼近算力屋顶。

In [ ]:
# 计算 attention 的算术强度,判断它是 memory-bound 还是 compute-bound
def attention_ai(N, H, d, dtype_bytes=2):
    """attention 的算术强度(FLOPs/字节)。"""
    flops = 4 * N * N * H * d                 # QK^T 与 PV 各约 2·N²·H·d
    bytes_ = (3 * N * H * d + N * H * d) * dtype_bytes   # QKV 读 + O 写(bf16)
    bytes_ += 2 * (H * N * N) * dtype_bytes   # S 与 P 各读写一次(naive 的 N² 访存)
    return flops / bytes_                     # 算术强度(FLOPs/字节)

N, H, d = 4096, 32, 128                       # LLaMA-7B 单层、单 batch
ai = attention_ai(N, H, d)
print(f"LLaMA-7B attention @N={N},H={H},d={d}:")
print(f"  算术强度 AI = {ai:.1f} FLOPs/字节")

# 对照不同硬件的 ridge 点
ridges = {"A100 (BF16)": 156, "H100 (BF16)": 295, "B200 (BF16)": 281}
print(f"\n当前 AI={ai:.0f} vs 各硬件 ridge(AI*):")
for name, r in ridges.items():
    bound = "compute-bound" if ai > r else "memory-bound"
    print(f"  {name:<14} ridge={r:>4}  → {bound}")

# FlashAttention 把 S/P 留在片上,访存去掉 N² 项 → AI 大幅提升
def flash_ai(N, H, d, dtype_bytes=2):
    flops = 4 * N * N * H * d
    bytes_ = (3 * N * H * d + N * H * d) * dtype_bytes   # 只 QKV 读 + O 写,无 S/P
    return flops / bytes_
print(f"\n若用 FlashAttention(去 S/P):AI ≈ {flash_ai(N,H,d):.0f} FLOPs/字节 → 显著右移,更接近 compute-bound")

## 6. 与 vLLM 的关系 🔗

memory-bound 这个判断对 vLLM 的工程决策有直接指导:

1. **后端选择**(第 47 课):正因为 attention 是 memory-bound,vLLM 才那么在意选对 kernel(FlashAttention-2 等),把 S/P 留在片上减少访存;
2. **FP8 KV cache**:把 KV cache 的字节数减半,直接翻倍 memory-bound 区间的算术强度,在 decode 阶段几乎线性提速——因为瓶颈就是带宽,不是算力;
3. **decode vs prefill**:decode 每步只算 1 个 token,算术强度 ≈ 1,深陷 memory-bound;prefill 一次处理整个 prompt,算术强度几百,接近 compute-bound。这就是为什么长序列训练/prefill 用 FlashAttention,而 decode 的瓶颈在搬 KV cache。

> 📄 参考:zeroentropy.dev「Arithmetic intensity: FLOPs per byte and the roofline model」、baseten.co「A guide to LLM inference and performance」给出同样的 roofline 视角。

## 7. 🖥️ Streamlit 动态演示:拖序列长度实测 🎛️

运行同目录下的 `app_46_attention_perf.py`,拖动**序列长度 / 头数**,程序会在你的 GPU 上**实时跑小实验**,对比三种实现的耗时,并验证误差:

```bash
python -m streamlit run app_46_attention_perf.py
```

浏览器打开 **http://localhost:8501**。建议把序列长度从 128 拖到 1024,观察红色(naive)曲线如何快速爬升。完整源码如下(与同目录 app 一字不差):

📜 这就是 app_46_attention_perf.py 的完整源码,notebook 与 app 共用同一套 bench / 三种实现,保证演示与讲解一致。

In [ ]:
%%writefile app_46_attention_perf.py
# -*- coding: utf-8 -*-
# app_46_attention_perf.py — Attention 性能对比:naive vs 分块 vs SDPA 🚀
import time
import numpy as np
import plotly.graph_objects as go
import streamlit as st
import torch
import torch.nn.functional as F

st.set_page_config(page_title="🚀 46 · Attention 性能", layout="wide")
st.title("🚀 第 46 课 · Attention 性能:naive vs 分块 vs torch SDPA")

st.markdown("""
三种 attention 实现,谁更快?用你的 **GPU** 实时跑一小段实验对比。naive 把 N×N 打分矩阵整体摊开;
分块(flash)用在线 softmax 边扫边累积;`F.scaled_dot_product_attention` 是 PyTorch 内置的 kernel 化
实现。拖动**序列长度 / 头数**,看耗时如何随参数变化。
""")

def naive_attention(Q, K, V):
    d = Q.shape[-1]
    S = torch.einsum("bhnd,bhmd->bhnm", Q, K) / (d ** 0.5)
    P = torch.softmax(S, dim=-1)
    return torch.einsum("bhnm,bhmd->bhnd", P, V)

def flash_chunked(Q, K, V, block_M=64):
    B, H, N, d = Q.shape
    O = torch.zeros_like(Q)
    m = torch.full((B, H, N, 1), float("-inf"))
    l = torch.zeros((B, H, N, 1))
    scale = d ** -0.5
    for j in range(0, N, block_M):
        Kj = K[:, :, j:j + block_M, :]; Vj = V[:, :, j:j + block_M, :]
        S = torch.einsum("bhnd,bhmd->bhnm", Q, Kj) * scale
        m_new = torch.maximum(m, S.max(dim=-1, keepdim=True).values)
        P = torch.exp(S - m_new)
        l_new = l * torch.exp(m - m_new) + P.sum(dim=-1, keepdim=True)
        O = O * torch.exp(m - m_new) + torch.einsum("bhnm,bhmd->bhnd", P, Vj)
        m, l = m_new, l_new
    return O / l

def bench(fn, *a, iters=3):
    fn(*a)
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(*a)
    return (time.perf_counter() - t0) / iters * 1000

with st.sidebar:
    st.header("🎛️ 参数")
    N = st.slider("序列长度 N", 128, 1024, 256, 64)
    H = st.slider("头数 H", 1, 8, 4, 1)
    d = st.selectbox("头维度 d", [32, 64], index=0)
    st.caption("实验在你的 GPU 上真实运行几轮,请等待 1-2 秒。")

torch.manual_seed(0)
Q = torch.randn(1, H, N, d); K = torch.randn(1, H, N, d); V = torch.randn(1, H, N, d)

t_n = bench(naive_attention, Q, K, V)
t_f = bench(flash_chunked, Q, K, V, 64)
t_s = bench(lambda a, b, c: F.scaled_dot_product_attention(a, b, c), Q, K, V)

c1, c2, c3, c4 = st.columns(4)
c1.metric("naive", f"{t_n:.2f} ms")
c2.metric("分块 flash", f"{t_f:.2f} ms")
c3.metric("torch SDPA", f"{t_s:.2f} ms")
c4.metric("SDPA vs naive 加速", f"{t_n/max(t_s,1e-9):.1f}×")

o1 = naive_attention(Q, K, V); o2 = F.scaled_dot_product_attention(Q, K, V)
st.metric("naive vs SDPA 最大误差", f"{float((o1-o2).abs().max()):.2e}")

Ns = list(range(128, 1025, 128))
tn, tf, ts = [], [], []
for n in Ns:
    q = torch.randn(1, H, n, d); k = torch.randn(1, H, n, d); v = torch.randn(1, H, n, d)
    tn.append(bench(naive_attention, q, k, v))
    tf.append(bench(flash_chunked, q, k, v, 64))
    ts.append(bench(lambda a, b, c: F.scaled_dot_product_attention(a, b, c), q, k, v))

fig = go.Figure()
for name, arr, col in [("naive", tn, "#E45756"), ("分块", tf, "#4C78A8"), ("SDPA", ts, "#72B7B2")]:
    fig.add_trace(go.Scatter(x=Ns, y=arr, name=name, mode="lines+markers",
                             line=dict(color=col, width=2.5)))
fig.update_layout(title=f"三种实现耗时 vs 序列长度(H={H}, d={d})",
                  xaxis_title="序列长度 N", yaxis_title="耗时(ms)", height=440,
                  legend=dict(orientation="h", y=1.12), margin=dict(l=10, r=10, t=60, b=10))
st.plotly_chart(fig, use_container_width=True)

st.caption("⭐ 观察:naive 随 N 二次方爬升;分块与 SDPA 明显更平缓。三种实现数值上几乎一致(误差<1e-6)。")

st.markdown("""
> 💡 **结论**:attention 是典型的 **memory-bound(访存受限)** 计算——瓶颈在把矩阵搬进搬出内存,
> 而不在计算本身。GPU 算力常常过剩,谁少搬数据谁就快。naive 多搬了 N×N 的 S/P,分块/SDPA 把它们
> 留在片上,于是快。**优化 attention 的钥匙是减少访存,而不是增加算力。**
""")
st.caption("《minivllm: 图解 vLLM 推理引擎》第 7 章 · 第 46 课配套演示")

## ✅ 本课小结

- 算术强度 AI=FLOPs/字节;当 AI < 硬件 ridge(AI*=算力/带宽)时为 memory-bound,否则 compute-bound
- naive attention 的 AI 只有 ~120(远低于 H100 ridge 295),是典型的 memory-bound 算子
- 同条件 GPU 真测:naive 随 N 约按 O(N²) 蹿升,SDPA 内核化最快;三者数值等价(误差<1e-6)
- torch 手写分块虽省显存,却因每次 block 都要 Python 循环 + 多次 kernel 启动,在真机上不一定快
- 增加算力对 memory-bound 算子帮助有限,减少中间张量读写(S/P)才是关键
- FlashAttention 把 S/P 留在片上,把 AI 提到几百,才逼近算力屋顶;FP8 KV cache 靠减半字节数直接提速
- decode 每步算术强度≈1,memory-bound;prefill 强度几百,compute-bound——两者要不同 kernel

## ✍️ 动手练习

1. 把 bench 的 iters 调大(如 20),比较计时噪声对结论的影响
2. 用 H=16, d=64 重跑对比,观察头数对三种实现耗时的影响
3. 推导 attention 的算术强度(FLOPs/Byte),画出它在 roofline 上的位置
4. 在 app_46 里加一个「随 d 变化」的扫描维度
5. 对比 decode(batch=1)与 prefill(长 N)的算术强度,解释为何 prefill 更接近 compute-bound

## 🔗 延伸阅读

- [Roofline 模型 (CACM 2009)](https://people.eecs.berkeley.edu/~pattrsn/mytalks/Roofline.pdf)
- [FlashAttention 论文](https://arxiv.org/abs/2205.14135)
- [PyTorch SDPA 文档](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

---
> 🎉 恭喜完成本课!下一课见!